# Teacher YOLO26m — InaTRC

Notebook mandiri untuk Kaggle/Colab. Semua fungsi dan algoritme terlihat di bawah.
Run All menjalankan **satu eksperimen**. Default notebook ini: **full**, seed 42.
Teacher/baseline/Native full memerlukan dataset asli; semua KD full memerlukan teacher YOLO26m InaTRC yang sama.
Smoke memakai data sintetis dan hasilnya **bukan hasil penelitian**. CrossKD/CSAKD berstatus **pilot**.

Protokol bersama: 640 px, batch 16, AdamW, maksimal 100 epoch, patience 25. Batch tidak diturunkan otomatis.
Metrik utama: mAP50–95 Ultralytics pada **val**; split test hanya diaudit.
Internet diperlukan saat dependency atau pretrained COCO belum tersedia.

## Konfigurasi Eksperimen
Isi path dataset/checkpoint. Pilih `smoke` untuk diagnosis 1 epoch sebelum training penelitian.

In [ ]:
# Stage/model/metode sesuai judul; gunakan notebook lain untuk eksperimen lain.
STAGE = 'teacher'
STUDENT = 'n'  # teacher selalu memakai YOLO26m
METHOD = 'none'
MODE = 'full'  # "smoke" atau "full"
SEED = 42
DEVICE = "0"  # GPU pertama; "cpu" hanya untuk diagnosis lokal

DATASET_ROOT = ""  # Kaggle: otomatis mencari satu dataset train/val/test
EXPECTED_IMAGES = 3250  # ubah eksplisit jika memakai revisi dataset berbeda
TEACHER_CKPT = ""  # KD full: /kaggle/input/.../best.pt atau path Colab
TEACHER_SHA256 = ""  # opsional: kunci hash teacher yang sama di semua KD full
MODEL_WEIGHTS = None  # opsional: pretrained COCO .pt lokal sesuai ukuran model
INPUT_ROOT = None  # otomatis: /kaggle/input atau /content/inputs
OUTPUT_ROOT = None  # otomatis: /kaggle/working/outputs atau /content/outputs

WANDB_MODE = "disabled"  # "disabled", "offline", atau "online"
WANDB_PROJECT = "skripsi-inatrc-yolo26"
WANDB_ENTITY = None  # username/team; WANDB_API_KEY di Secrets, jangan ditulis di sini

# Fitur tambahan: samakan pilihan COCO di semua eksperimen yang dibandingkan.
EVALUATE_COCO_AREA = False  # True: AP-S/AP-M/AP-L tambahan, lebih banyak ekspor/evaluasi
CREATE_RESULTS_ZIP = False  # True: bundel unduhan; Kaggle Save Version sudah menyimpan folder

# Opsional Colab Drive: uncomment, lalu isi DATASET_ROOT / TEACHER_CKPT di atas.
# from google.colab import drive
# drive.mount("/content/drive")

## Instalasi Dependency
Pip menggunakan interpreter kernel. Rencana instalasi diperiksa terlebih dahulu; Torch/torchvision bawaan dipertahankan. Bila paket sudah di-import dengan versi berbeda, restart session/kernel lalu Run All.

In [ ]:
"""Helper yang ditampilkan langsung di notebook; runtime tidak mengimpor file ini."""

import importlib.metadata

import json

import os

from pathlib import Path

import re

import subprocess

import sys

import time

import uuid

import zipfile

def _setup_redact(text):
    """Jangan tulis credential ke layar atau log instalasi."""
    value = str(text)
    key = os.environ.get("WANDB_API_KEY", "")
    if key:
        value = value.replace(key, "[REDACTED]")
    return value

def _setup_run_command(command, log_path):
    """Tampilkan progres pip sambil menyimpan log yang dapat diunduh."""
    chunks = []
    with Path(log_path).open("a", encoding="utf-8") as log:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace",
        )
        for line in process.stdout:
            safe_line = _setup_redact(line)
            print(safe_line, end="", flush=True)
            log.write(safe_line)
            log.flush()
            chunks.append(safe_line)
        returncode = process.wait()
    if returncode:
        tail = "".join(chunks)[-2000:]
        raise RuntimeError(f"pip gagal (exit {returncode}). {tail}\nLog: {log_path}")
    return "".join(chunks)

def _setup_package_snapshot(package):
    distribution = importlib.metadata.distribution(package)
    return {
        "version": distribution.version,
        "location": str(Path(distribution.locate_file("")).resolve()),
    }

def validate_loaded_packages(packages):
    """Package yang diganti pip harus dimuat ulang melalui restart kernel."""
    module_names = {
        "pillow": "PIL", "pyyaml": "yaml", "faster-coco-eval": "faster_coco_eval",
        "opencv-python": "cv2", "opencv-python-headless": "cv2",
        "ultralytics-thop": "thop",
    }
    for package in packages:
        normalized = re.sub(r"[-_.]+", "-", package).lower()
        module_name = module_names.get(normalized, normalized.replace("-", "_"))
        module = sys.modules.get(module_name)
        if module is None:
            continue
        installed = _setup_package_snapshot(package)
        loaded_version = getattr(module, "__version__", None)
        module_file = getattr(module, "__file__", None)
        # cv2 tidak menampilkan nomor build wheel, tetapi versi core harus sama.
        expected_version, active_version = installed["version"], str(loaded_version)
        if normalized in {"opencv-python", "opencv-python-headless"}:
            expected_core = re.match(r"^\d+\.\d+\.\d+", expected_version)
            active_core = re.match(r"^\d+\.\d+\.\d+", active_version)
            expected_version = expected_core.group() if expected_core else expected_version
            active_version = active_core.group() if active_core else active_version
        stale_version = loaded_version is not None and active_version != expected_version
        stale_location = module_file is not None and not Path(module_file).resolve().is_relative_to(
            Path(installed["location"]).resolve()
        )
        if stale_version or stale_location:
            raise RuntimeError(
                f"{package} yang sudah dimuat berbeda dari instalasi aktif. "
                "Restart Session (Kaggle) / Restart runtime (Colab), lalu Run All kembali."
            )

def _setup_pinned_requirements(requirements):
    pins = {}
    for requirement in requirements:
        match = re.fullmatch(r"([A-Za-z0-9][A-Za-z0-9_.-]*)==([A-Za-z0-9][A-Za-z0-9_.+!-]*)", requirement)
        if match is None:
            raise ValueError(f"Dependency notebook harus berupa nama==versi: {requirement!r}")
        name, version = match.groups()
        normalized = re.sub(r"[-_.]+", "-", name).lower()
        if normalized in {"torch", "torchvision"}:
            raise ValueError("Torch/torchvision berasal dari runtime; jangan masukkan ke dependency notebook.")
        if normalized in pins and pins[normalized][1] != version:
            raise ValueError(f"Pin dependency ganda berbeda: {name}")
        pins[normalized] = (name, version)
    return pins

def _setup_validate_requirements(packages):
    """Periksa dependency aplikasi aktif; tool lain bawaan Kaggle tidak dilatih."""
    from packaging.requirements import Requirement

    checked, visited = [], set()
    pending = list(packages)
    # Periksa dependency transitif juga sebelum memutuskan melewati resolver.
    for package in pending:
        normalized = re.sub(r"[-_.]+", "-", package).lower()
        if normalized in visited:
            continue
        visited.add(normalized)
        for declaration in importlib.metadata.requires(package) or []:
            requirement = Requirement(declaration)
            if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
                continue
            try:
                installed = importlib.metadata.version(requirement.name)
            except importlib.metadata.PackageNotFoundError:
                raise RuntimeError(f"Dependency {package} belum terpenuhi: {requirement.name}.") from None
            if not requirement.specifier.contains(installed, prereleases=True):
                raise RuntimeError(f"Dependency {package} tidak sesuai: {declaration}; aktif {installed}.")
            checked.append({"package": package, "requirement": declaration, "active_version": installed})
            pending.append(requirement.name)
    return checked

def setup_dependencies(requirements, output_root):
    """Resolve dahulu, lindungi Torch CUDA, lalu install paket aplikasi saja."""
    if not (3, 11) <= sys.version_info[:2] < (3, 14):
        raise RuntimeError("Notebook mendukung Python kernel 3.11, 3.12, dan 3.13.")
    pins = _setup_pinned_requirements(requirements)
    logs = Path(output_root).expanduser().resolve() / "setup_logs"
    logs.mkdir(parents=True, exist_ok=True)
    log_path = logs / "pip_setup.log"
    report_path = logs / "pip_plan.json"
    constraints = logs / "protected_torch.txt"
    started = time.perf_counter()
    print("Instalasi dependency: memeriksa versi dan kompatibilitas...", flush=True)
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\n=== Instalasi dependency ===\n")
    try:
        before = {name: _setup_package_snapshot(name) for name in ("torch", "torchvision")}
        validate_loaded_packages(["torch", "torchvision"])
        # Version yang cocok belum cukup: dependency graph juga harus valid.
        checked = None
        try:
            available = {name: _setup_package_snapshot(name) for name, _ in pins.values()}
            if all(available[name]["version"] == version for name, version in pins.values()):
                checked = _setup_validate_requirements(["torch", "torchvision", *available])
        except (importlib.metadata.PackageNotFoundError, RuntimeError):
            pass  # Paket hilang/konflik: resolver pip tetap diperlukan.
        if checked is not None:
            report = {"install": [], "resolution_skipped": True}
            report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
        else:
            constraints.write_text(
                "".join(f"{name}=={info['version']}\n" for name, info in before.items()), encoding="utf-8"
            )
            # Report hanya memuat rencana; belum ada package yang diubah di sini.
            command = [
                sys.executable, "-m", "pip", "install", "--dry-run", "--report", str(report_path),
                "--constraint", str(constraints), "--only-binary=faster-coco-eval",
                "--disable-pip-version-check", *[f"{name}=={version}" for name, version in pins.values()],
            ]
            _setup_run_command(command, log_path)
            report = json.loads(report_path.read_text(encoding="utf-8"))
            # Simpan versi report yang sudah disanitasi, bukan environment/credential.
            report_path.write_text(_setup_redact(json.dumps(report, indent=2)), encoding="utf-8")
        install_pins = []
        for item in report.get("install", []):
            metadata = item["metadata"]
            name, version = metadata["name"], metadata["version"]
            normalized = re.sub(r"[-_.]+", "-", name).lower()
            if normalized in before:
                raise RuntimeError(
                    f"STOP: pip hendak mengganti {name}. Torch/torchvision bawaan tidak boleh diubah."
                )
            _setup_pinned_requirements([f"{name}=={version}"])
            install_pins.append(f"{name}=={version}")
        if install_pins:
            print("Instalasi dependency: memasang paket aplikasi yang telah di-resolve...", flush=True)
            _setup_run_command([
                sys.executable, "-m", "pip", "install", "--no-deps",
                "--only-binary=faster-coco-eval", "--disable-pip-version-check", *install_pins,
            ], log_path)
        else:
            print("Dependency sudah sesuai; instalasi dilewati.", flush=True)
        after = {name: _setup_package_snapshot(name) for name in before}
        if after != before:
            raise RuntimeError("STOP: versi/lokasi Torch berubah. Pulihkan runtime GPU sebelum melanjutkan.")
        installed = {name: _setup_package_snapshot(name) for name, _ in pins.values()}
        for name, version in pins.values():
            if installed[name]["version"] != version:
                raise RuntimeError(f"Versi {name} tidak sesuai pin {version}.")
        validate_loaded_packages(["torch", "torchvision", *installed])
        # --no-deps tetap harus menghasilkan dependency graph aplikasi yang konsisten.
        if checked is None:
            checked = _setup_validate_requirements(["torch", "torchvision", *installed])
        elapsed = round(time.perf_counter() - started, 3)
        print(f"Instalasi dependency selesai: {elapsed:.1f} detik. Log: {log_path}", flush=True)
        with log_path.open("a", encoding="utf-8") as log:
            log.write(f"Selesai: {elapsed:.3f} detik\n")
        return {
            "python_executable": sys.executable, "torch_before": before, "torch_after": after,
            "packages": installed, "duration_seconds": elapsed,
            "dependency_checks": checked,
            "log_path": str(log_path), "report_path": str(report_path),
        }
    except Exception as exc:
        safe_error = _setup_redact(f"{type(exc).__name__}: {exc}")
        elapsed = time.perf_counter() - started
        with log_path.open("a", encoding="utf-8") as log:
            log.write(f"GAGAL setelah {elapsed:.3f} detik: {safe_error}\n")
        raise RuntimeError(
            f"Tahap Instalasi Dependency gagal setelah {elapsed:.1f} detik. "
            f"{safe_error}\nLog: {log_path}"
        ) from None

def detect_platform():
    """Gunakan direktori platform; smoke lokal boleh memakai CPU."""
    if Path("/kaggle/working").is_dir():
        return {
            "name": "kaggle", "work_root": "/kaggle/working", "input_root": "/kaggle/input",
            "output_root": "/kaggle/working/outputs",
        }
    if Path("/content").is_dir() or "google.colab" in sys.modules:
        return {
            "name": "colab", "work_root": "/content", "input_root": "/content/inputs",
            "output_root": "/content/outputs",
        }
    return {
        "name": "local", "work_root": str(Path.cwd()), "input_root": None,
        "output_root": str(Path.cwd() / "outputs"),
    }

def get_wandb_secret(mode, platform):
    """Ambil key hanya untuk online; nilai key tidak pernah dikembalikan/dicetak."""
    if mode not in {"disabled", "offline", "online"}:
        raise ValueError("WANDB_MODE harus disabled, offline, atau online.")
    if mode != "online":
        os.environ.pop("WANDB_API_KEY", None)
        return None
    if os.environ.get("WANDB_API_KEY"):
        return None
    name = platform["name"] if isinstance(platform, dict) else platform
    try:
        if name == "kaggle":
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("WANDB_API_KEY")
        elif name == "colab":
            from google.colab import userdata
            key = userdata.get("WANDB_API_KEY")
        else:
            raise ValueError("WANDB_API_KEY belum diatur pada environment lokal.")
        if not isinstance(key, str) or not key.strip():
            raise ValueError("Secret kosong.")
        os.environ["WANDB_API_KEY"] = key
    except Exception:
        raise RuntimeError(
            "W&B online memerlukan WANDB_API_KEY dari environment atau Secrets platform. "
            "Aktifkan akses secret, atau pilih WANDB_MODE='disabled'/'offline'."
        ) from None
    return None

def _standalone_is_dataset_root(path):
    path = Path(path)
    return all((path / split / kind).is_dir() for split in ("train", "val", "test") for kind in ("images", "labels"))

def discover_dataset_root(root, input_root, mode):
    """Pilih hanya satu dataset dengan ketiga split; jangan menebak jika ambigu."""
    if mode not in {"smoke", "full"}:
        raise ValueError("MODE harus smoke atau full.")
    if root:
        selected = Path(root).expanduser().resolve()
        if not _standalone_is_dataset_root(selected):
            raise RuntimeError(f"DATASET_ROOT harus berisi train/val/test dengan images dan labels: {selected}")
        return str(selected)
    candidates = set()
    if input_root and Path(input_root).is_dir():
        inputs = Path(input_root).expanduser().resolve()
        for directory, subdirs, _ in os.walk(inputs):
            # Lewati gambar individual; penemuan hanya membutuhkan direktori split.
            subdirs[:] = sorted(name for name in subdirs if name not in {"images", "labels", ".git", ".cache"})
            candidate = Path(directory)
            if _standalone_is_dataset_root(candidate):
                candidates.add(candidate.resolve())
                subdirs[:] = []
    if len(candidates) > 1:
        paths = ", ".join(str(path) for path in sorted(candidates))
        raise RuntimeError(f"Lebih dari satu dataset ditemukan. Isi DATASET_ROOT secara eksplisit: {paths}")
    if candidates:
        return str(next(iter(candidates)))
    if mode == "full":
        raise RuntimeError("Dataset InaTRC belum ditemukan. Tambahkan Kaggle Input atau isi DATASET_ROOT.")
    return None

def create_results_zip(run_root, output_root=None):
    """Bundle hanya hasil run ini; data sintetis/secret/teacher tidak ikut."""
    run = Path(run_root).expanduser().resolve()
    if not run.is_dir():
        raise RuntimeError(f"Direktori hasil tidak tersedia: {run}")
    target = Path(output_root).expanduser().resolve() if output_root else run.parent
    target.mkdir(parents=True, exist_ok=True)
    archive_path = target / f"{run.name}-results-{uuid.uuid4().hex[:8]}.zip"
    approved_names = {
        "best.pt", "last.pt", "results.csv", "epoch_history.csv", "checkpoint_manifest.csv",
        "metrics.json", "metrics.csv", "per_class.csv", "coco_metrics.json", "coco_metrics.csv",
        "resolved_config.json", "resolved_config.yaml", "dataset_audit.json", "dataset_fingerprint.json",
        "environment.json", "metadata.json", "checkpoint_audit.json", "summary.json", "summary.csv",
        "source_provenance.json", "training_status.json", "initial_weights.json",
        "args.yaml", "complexity.json", "run_summary.json",
        "training_summary.json", "coco_area.json", "initial_provenance.json", "teacher_provenance.json",
        "data_runtime.yaml", "predictions.json",
        "dataset_audit.csv", "class_distribution.csv", "bbox_distribution.csv", "cross_split_exact_duplicates.csv",
    }
    skipped_parts = {"wandb", "synthetic_data", "synthetic", "dataset", "teacher", "diagnostic_teacher", "cache", ".cache"}
    selected = []
    for path in sorted(run.rglob("*")):
        if not path.is_file() or path.is_symlink() or not path.resolve().is_relative_to(run):
            continue
        relative = path.relative_to(run)
        if any(part in skipped_parts for part in relative.parts):
            continue
        # Kurva dan matriks validasi ikut sebagai hasil akhir.
        is_plot = path.suffix.lower() in {".png", ".jpg", ".svg", ".pdf"} and (
            any(part in {"val", "validation", "best-val", "evaluation"} for part in relative.parts)
            or path.stem.endswith("curve")
            or path.stem.startswith(("confusion_matrix", "results"))
        )
        if path.name not in approved_names and not is_plot:
            continue
        if path.suffix.lower() in {".json", ".yaml", ".csv"}:
            content = path.read_text(encoding="utf-8")
            if "WANDB_API_KEY" in content or os.environ.get("WANDB_API_KEY", "\0") in content:
                raise RuntimeError(f"Bundle dibatalkan: credential ditemukan dalam {relative}.")
        selected.append((path, relative))
    if not selected:
        raise RuntimeError("Tidak ada artifact hasil yang dikenali untuk dibundel.")
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path, relative in selected:
            archive.write(path, arcname=str(Path(run.name) / relative))
    print(f"Bundle hasil: {archive_path}", flush=True)
    return archive_path

In [ ]:
APP_REQUIREMENTS = ['ultralytics==8.4.155',
 'wandb==0.28.1',
 'faster-coco-eval==1.6.7',
 'pandas==2.3.3',
 'pillow==12.3.0',
 'pyyaml==6.0.3',
 'matplotlib==3.11.2',
 'ultralytics-thop==2.2.2',
 'packaging==26.3',
 'opencv-python==5.0.0.93']

APP_REQUIREMENTS.append("numpy==" + ('2.4.6' if sys.version_info[:2] < (3, 12) else '2.5.3'))
if not EVALUATE_COCO_AREA:
    APP_REQUIREMENTS = [pin for pin in APP_REQUIREMENTS if not pin.startswith("faster-coco-eval==")]
PLATFORM_INFO = detect_platform()
INPUT_ROOT = str(Path(INPUT_ROOT or PLATFORM_INFO["input_root"] or Path.cwd() / "inputs").expanduser().resolve())
OUTPUT_ROOT = str(Path(OUTPUT_ROOT or PLATFORM_INFO["output_root"]).expanduser().resolve())
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
print("Platform:", PLATFORM_INFO["name"], "| Output:", OUTPUT_ROOT, flush=True)
SETUP_REPORT = setup_dependencies(APP_REQUIREMENTS, OUTPUT_ROOT)
get_wandb_secret(WANDB_MODE, PLATFORM_INFO)

In [ ]:
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO, settings
from ultralytics.cfg import get_cfg
from ultralytics.utils.torch_utils import init_seeds, unwrap_model

# Matplotlib menulis cache di output, terpisah dari dataset sumber.
os.environ.setdefault("MPLCONFIGDIR", str(Path(OUTPUT_ROOT) / "matplotlib-cache"))

## Pemeriksaan GPU
Cek versi aktif, operasi CUDA, torchvision NMS, dan pertukaran NumPy–Torch. Gagal di sini menghentikan training.

In [ ]:
import hashlib

import importlib.metadata

import json

import platform

import sys

from pathlib import Path

from typing import Any

INATRC_NAMES = [
    "Kelas-1 (Non-truk)",
    "Kelas-2 (Truk 2 sumbu)",
    "Kelas-3 (Truk 3 sumbu)",
    "Kelas-4 (Truk 4 sumbu)",
    "Kelas-5 (Truk >=5 sumbu)",
]

def _device(device: str, require_gpu: bool = False):
    import torch

    requested = str(device).lower()
    if requested == "auto":
        requested = "cuda:0" if torch.cuda.is_available() else "cpu"
    elif requested.isdecimal():
        requested = f"cuda:{requested}"
    selected = torch.device(requested)
    if require_gpu and selected.type != "cuda":
        raise RuntimeError("A CUDA GPU is required; select device='0' or 'cuda:0'.")
    if selected.type == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError(
                f"Requested GPU device {device!r}, but torch.cuda.is_available() is False. "
                f"Active Python: {sys.executable}; Torch: {torch.__version__}; CUDA: {torch.version.cuda}."
            )
        index = selected.index if selected.index is not None else 0
        if index >= torch.cuda.device_count():
            raise RuntimeError(f"CUDA device {index} does not exist ({torch.cuda.device_count()} available).")
        selected = torch.device("cuda", index)
    elif selected.type != "cpu":
        raise ValueError("Stage A supports a single CPU or CUDA device: 'cpu', '0', or 'cuda:0'.")
    return selected

def _validate_runtime_requirements(packages=("torch", "torchvision")) -> list[dict[str, str]]:
    """Check active mandatory Requires-Dist entries, including a borrowed Kaggle CUDA stack."""
    from packaging.requirements import Requirement

    checked = []
    for package in packages:
        try:
            requirements = importlib.metadata.requires(package) or []
        except importlib.metadata.PackageNotFoundError as exc:
            raise RuntimeError(f"Cannot inspect runtime dependency metadata for {package!r}.") from exc
        for declaration in requirements:
            requirement = Requirement(declaration)
            if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
                continue
            try:
                active_version = importlib.metadata.version(requirement.name)
            except importlib.metadata.PackageNotFoundError as exc:
                raise RuntimeError(
                    f"Active {package} Requires-Dist {declaration!r} is not satisfied: "
                    f"{requirement.name} is missing. Reconcile the pip/kernel environment dependencies "
                    "without automatically replacing the inherited CUDA Torch stack."
                ) from exc
            if requirement.specifier and not requirement.specifier.contains(active_version, prereleases=True):
                raise RuntimeError(
                    f"Active {package} Requires-Dist {declaration!r} is not satisfied: "
                    f"active {requirement.name}=={active_version}. Reconcile the pip/kernel environment "
                    "dependencies without automatically replacing the inherited CUDA Torch stack."
                )
            checked.append({"package": package, "requirement": declaration, "active_version": active_version})
    return checked

def _validate_python_torch_pair(torch_version: str, vision_version: str, python_version=None):
    """Periksa pasangan resmi Torch/torchvision beserta dukungan versi Python."""
    from packaging.version import Version

    python_version = tuple(python_version or sys.version_info[:2])
    if not (3, 11) <= python_version < (3, 14):
        raise RuntimeError(f"Supported Python versions are 3.11, 3.12 and 3.13; found {python_version}.")
    torch_release, vision_release = Version(torch_version), Version(vision_version)
    torch_pair, vision_pair = torch_release.release[:2], vision_release.release[:2]
    if (torch_release.is_prerelease or torch_release.is_devrelease
            or vision_release.is_prerelease or vision_release.is_devrelease
            or torch_pair[0] != 2 or not 4 <= torch_pair[1] <= 13
            or vision_pair != (0, torch_pair[1] + 15)):
        raise RuntimeError(
            f"Unsupported Torch/torchvision pair: {torch_version}/{vision_version}. "
            "Require stable Torch 2.4 through 2.13 with torchvision 0.19 through 0.28 respectively."
        )
    # Tabel resmi torchvision membatasi pasangan 2.4–2.6 sampai Python 3.12.
    if python_version >= (3, 13) and torch_pair < (2, 7):
        raise RuntimeError(
            f"Python 3.13 requires Torch >=2.7 and torchvision >=0.22; found {torch_version}/{vision_version}. "
            "Use a compatible preinstalled runtime; do not automatically replace CUDA wheels."
        )
    return torch_pair, vision_pair

def environment_report(device: str = "cpu", require_gpu: bool = False,
                       dependency_checks: list[dict] | None = None) -> dict[str, Any]:
    """Check the active interpreter, Torch/CUDA, and torchvision NMS on the requested device."""
    import torch
    selected = _device(device, require_gpu)
    try:
        import torchvision
    except Exception as exc:
        raise RuntimeError(
            f"torchvision cannot load with active Torch {torch.__version__} from {torch.__file__}. "
            "Check the Torch/torchvision version pair and wheel build."
        ) from exc
    versions = {}
    for package in ("ultralytics", "wandb", "torch", "torchvision", "numpy", "PyYAML"):
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    torch_pair, vision_pair = _validate_python_torch_pair(torch.__version__, torchvision.__version__)
    for package, expected in (("ultralytics", "8.4.155"), ("wandb", "0.28.1")):
        if versions[package] != expected:
            raise RuntimeError(f"Expected {package}=={expected}; active version is {versions[package]!r}.")
    requirement_checks = _validate_runtime_requirements() if dependency_checks is None else dependency_checks
    report: dict[str, Any] = {
        "python": platform.python_version(),
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "torch_file": torch.__file__,
        "torchvision_version": torchvision.__version__,
        "torchvision_file": torchvision.__file__,
        "package_versions": versions,
        "cuda_runtime": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "requested_device": str(device),
        "resolved_device": str(selected),
        "torchvision_pair_compatible": True,
        "dependency_check_passed": True,
        "runtime_dependency_checks": requirement_checks,
        "torch_differs_from_local_cpu_lock": torch_pair != (2, 10) or vision_pair != (0, 25),
        "torch_runtime_policy": (
            "kaggle_inherited_cuda_pair" if Path("/kaggle/working").exists()
            else "local_cpu_lock" if torch_pair == (2, 10) and vision_pair == (0, 25)
            else "compatible_alternate_pair"
        ),
        "gpus": [],
    }
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            properties = torch.cuda.get_device_properties(index)
            report["gpus"].append(
                {
                    "index": index,
                    "name": properties.name,
                    "vram_gib": round(properties.total_memory / 2**30, 3),
                    "compute_capability": [properties.major, properties.minor],
                }
            )
    try:
        # Jalankan operasi sungguhan; CUDA terdeteksi saja belum menjamin wheel cocok.
        import numpy as np
        probe = torch.tensor([1.0, 2.0], device=selected)
        result = float(probe.square().sum().cpu())
        boxes = torch.tensor([[0.0, 0.0, 2.0, 2.0], [0.0, 0.0, 1.0, 1.0]], device=selected)
        scores = torch.tensor([0.9, 0.8], device=selected)
        kept = torchvision.ops.nms(boxes, scores, 0.5).cpu().tolist()
        if result != 5.0 or kept != [0, 1]:
            raise RuntimeError(f"Unexpected tensor/NMS output: {result}, {kept}")
        # Loader memakai NumPy ↔ Torch; periksa ABI setelah dependency venv terpasang.
        array = np.array([1.0, 2.0], dtype=np.float32)
        roundtrip = torch.from_numpy(array).to(selected).square().cpu().numpy()
        if not np.array_equal(roundtrip, np.array([1.0, 4.0], dtype=np.float32)):
            raise RuntimeError("Unexpected NumPy/Torch roundtrip output")
        if selected.type == "cuda":
            torch.cuda.synchronize(selected)
        report["tensor_probe_passed"] = True
        report["torchvision_nms_passed"] = True
        report["numpy_torch_bridge_passed"] = True
    except Exception as exc:
        raise RuntimeError(
            f"Torch/torchvision/NumPy operation check failed on {selected}. "
            f"Torch {torch.__version__}, torchvision {torchvision.__version__}, "
            f"NumPy {versions['numpy']}, CUDA {torch.version.cuda}."
        ) from exc
    return report

import hashlib

import json

import subprocess

from pathlib import Path

def write_json(path: str | Path, value: object) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    return path

def sha256(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
validate_loaded_packages([requirement.split("==", 1)[0] for requirement in APP_REQUIREMENTS])
checked = None if SETUP_REPORT is None else [row for row in SETUP_REPORT["dependency_checks"] if row["package"] in ("torch", "torchvision")]
ENVIRONMENT = environment_report(str(DEVICE), require_gpu=str(DEVICE) != "cpu", dependency_checks=checked)
write_json(Path(OUTPUT_ROOT) / "setup_logs" / "environment.json", ENVIRONMENT)
print("Pemeriksaan GPU/ABI lolos:", ENVIRONMENT["resolved_device"], ENVIRONMENT["torch_version"], flush=True)

## Penemuan dan Audit Dataset
Audit membaca ketiga split, pasangan gambar/label, isi label, gambar rusak, dan duplikasi. Fingerprint memakai isi file. Kebocoran temporal/perseptual belum diperiksa. Cache loader hanya ditulis ke output.

In [ ]:
"""Audit gambar/label tanpa mengubah sumber; loader menyimpan cache di output.

Data sintetis hanya untuk smoke. Ultralytics dimuat saat loader diperlukan.
"""

import csv

import hashlib

import json

import math

import random

from collections import defaultdict

from pathlib import Path

from typing import TYPE_CHECKING, Any, Sequence

CLASS_NAMES = (
    "Kelas-1 (Non-truk)",
    "Kelas-2 (Truk 2 sumbu)",
    "Kelas-3 (Truk 3 sumbu)",
    "Kelas-4 (Truk 4 sumbu)",
    "Kelas-5 (Truk >=5 sumbu)",
)

SPLITS = ("train", "val", "test")

IMAGE_EXTENSIONS = frozenset({".jpg", ".jpeg", ".png", ".bmp", ".webp"})

LABEL_METADATA = frozenset({"classes.txt", "_classes.txt"})

BBOX_TOLERANCE = 1e-4

class DatasetAuditError(ValueError):
    """The dataset cannot be consumed without changing or excluding data."""

def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def _outside_source(path: Path, source: Path) -> Path:
    resolved = path.expanduser().resolve()
    if resolved == source or source in resolved.parents:
        raise DatasetAuditError(f"Output must be outside the source dataset: {resolved}")
    return resolved

def _files(directory: Path, *, labels: bool) -> list[Path]:
    if not directory.is_dir():
        raise DatasetAuditError(f"Missing required dataset directory: {directory}")
    result = []
    for path in sorted(directory.iterdir()):
        if path.is_dir():
            raise DatasetAuditError(f"Nested dataset directories are unsupported: {path}")
        if not path.is_file():
            raise DatasetAuditError(f"Unsupported dataset entry: {path}")
        if labels and path.name.lower() in LABEL_METADATA:
            continue
        supported = path.suffix.lower() == ".txt" if labels else path.suffix.lower() in IMAGE_EXTENSIONS
        if not supported:
            raise DatasetAuditError(f"Unsupported {'label' if labels else 'image'} file: {path}")
        result.append(path)
    return result

def _by_stem(paths: Sequence[Path], kind: str) -> dict[str, Path]:
    result: dict[str, Path] = {}
    for path in paths:
        if path.stem in result:
            raise DatasetAuditError(f"Duplicate {kind} stem: {result[path.stem]} and {path}")
        result[path.stem] = path
    return result

def _verify_image(path: Path) -> None:
    from PIL import Image

    try:
        with Image.open(path) as image:
            if image.width < 10 or image.height < 10:
                raise DatasetAuditError(f"Image dimensions must be at least 10 pixels: {path}")
            image_format = image.format
            image.verify()
        # verify() memeriksa struktur file; load() memastikan piksel bisa dibaca.
        with Image.open(path) as image:
            image.load()
        if image_format == "JPEG":
            with path.open("rb") as stream:
                stream.seek(-2, 2)
                if stream.read() != b"\xff\xd9":
                    raise DatasetAuditError(
                        f"JPEG EOF marker is missing; automatic JPEG repair is forbidden: {path}"
                    )
    except DatasetAuditError:
        raise
    except Exception as exc:
        raise DatasetAuditError(f"Corrupt or unreadable image {path}: {exc}") from exc

def _label_rows(path: Path, *, class_count: int = 5) -> tuple[bytes, list[tuple[float, ...]]]:
    try:
        raw = path.read_bytes()
        lines = raw.decode("utf-8-sig").strip().splitlines()
    except (OSError, UnicodeError) as exc:
        raise DatasetAuditError(f"Unreadable UTF-8 label {path}: {exc}") from exc
    rows = []
    seen: set[tuple[float, ...]] = set()
    for line_number, line in enumerate(lines, 1):
        location = f"{path}:{line_number}"
        parts = line.split()
        if len(parts) != 5:
            format_note = " Polygon/segmentation conversion requires explicit approval." if len(parts) > 5 else ""
            raise DatasetAuditError(
                f"{location}: expected YOLO bbox with 5 columns, found {len(parts)}.{format_note}"
            )
        try:
            row = tuple(float(part) for part in parts)
        except ValueError as exc:
            raise DatasetAuditError(f"{location}: nonnumeric label value") from exc
        if not all(math.isfinite(value) for value in row):
            raise DatasetAuditError(f"{location}: non-finite label value")
        cls, x, y, width, height = row
        # Format YOLO: class_id, pusat x/y, lebar/tinggi; koordinat relatif 0..1.
        if not cls.is_integer() or not 0 <= cls < class_count:
            raise DatasetAuditError(f"{location}: invalid class ID {cls}; expected 0..{class_count - 1}")
        if not (
            0 <= x <= 1 and 0 <= y <= 1 and 0 < width <= 1 and 0 < height <= 1
            and x - width / 2 >= -BBOX_TOLERANCE
            and y - height / 2 >= -BBOX_TOLERANCE
            and x + width / 2 <= 1 + BBOX_TOLERANCE
            and y + height / 2 <= 1 + BBOX_TOLERANCE
        ):
            raise DatasetAuditError(f"{location}: invalid normalized bounding box {row[1:]}")
        if row in seen:
            raise DatasetAuditError(f"{location}: duplicate label row; automatic removal is forbidden")
        seen.add(row)
        rows.append(row)
    return raw, rows

def _write_csv(path: Path, rows: Sequence[dict[str, Any]], fields: Sequence[str]) -> None:
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

def audit_dataset(
    root: str | Path,
    output_dir: str | Path | None = None,
    expected_images: int | None = 3250,
    class_names: Sequence[str] = CLASS_NAMES,
) -> dict[str, Any]:
    """Validate all original splits without editing, repairing, or dropping data.

    The fingerprint reproduces the baseline notebook's byte serialization,
    with streaming image hashes. ``expected_images=None`` disables only the
    version-specific image-count assertion. Exact duplicates are reported as
    warnings; this audit cannot establish temporal/perceptual independence.
    """
    root = Path(root).expanduser().resolve()
    if len(class_names) != 5 or any(not isinstance(name, str) or not name for name in class_names):
        raise DatasetAuditError("InaTRC requires exactly five nonempty class names")
    if expected_images is not None and (
        isinstance(expected_images, bool) or not isinstance(expected_images, int) or expected_images <= 0
    ):
        raise DatasetAuditError("expected_images must be a positive integer or None")
    destination = _outside_source(Path(output_dir), root) if output_dir is not None else None
    split_rows, class_rows, box_rows = [], [], []
    file_sha256 = {}
    occurrences: dict[str, list[dict[str, str]]] = defaultdict(list)
    fingerprint = hashlib.sha256()

    for split in SPLITS:
        images = _files(root / split / "images", labels=False)
        labels = _files(root / split / "labels", labels=True)
        image_map = _by_stem(images, "image")
        label_map = _by_stem(labels, "label")
        missing = sorted(image_map.keys() - label_map.keys())
        orphan = sorted(label_map.keys() - image_map.keys())
        if missing or orphan:
            raise DatasetAuditError(f"{split}: missing labels={missing}; orphan labels={orphan}")
        if not images:
            raise DatasetAuditError(f"Dataset split contains no images: {root / split}")
        counts = [0] * 5
        empty = 0
        for stem, image in sorted(image_map.items()):
            label = label_map[stem]
            _verify_image(image)
            image_hash = _sha256_file(image)
            label_bytes, rows = _label_rows(label)
            file_sha256[image.relative_to(root).as_posix()] = image_hash
            file_sha256[label.relative_to(root).as_posix()] = hashlib.sha256(label_bytes).hexdigest()
            # Fingerprint berubah jika nama, isi gambar, atau anotasi berubah.
            fingerprint.update(f"{split}/{image.name}\n{image_hash}\n{label.name}\n".encode())
            fingerprint.update(label_bytes)
            occurrences[image_hash].append({"split": split, "image": image.name})
            empty += int(not rows)
            for cls, _, _, width, height in rows:
                class_id = int(cls)
                counts[class_id] += 1
                box_rows.append({
                    "split": split, "class_id": class_id, "class_name": class_names[class_id],
                    "normalized_area": width * height,
                    "normalized_width": width, "normalized_height": height,
                })
        split_rows.append({"split": split, "images": len(images), "labels": len(labels),
                           "boxes": sum(counts), "empty_labels": empty})
        for class_id, count in enumerate(counts):
            class_rows.append({"split": split, "class_id": class_id,
                               "class_name": class_names[class_id], "boxes": count})

    total_images = sum(row["images"] for row in split_rows)
    if expected_images is not None and total_images != expected_images:
        raise DatasetAuditError(f"Expected {expected_images} images, found {total_images} at {root}")
    duplicates = [{"sha256": digest, "occurrences": entries}
                  for digest, entries in sorted(occurrences.items())
                  if len({entry["split"] for entry in entries}) > 1]
    warning_messages = []
    if duplicates:
        warning_messages.append(
            f"Found {len(duplicates)} exact cross-split duplicate groups; investigate before the final study."
        )
    report = {
        "source_path": str(root), "class_names": list(class_names), "expected_images": expected_images,
        "total_images": total_images, "total_boxes": len(box_rows),
        "sha256": fingerprint.hexdigest(), "fingerprint": fingerprint.hexdigest(),
        "file_sha256": file_sha256,
        "splits": split_rows, "class_distribution": class_rows, "bbox_distribution": box_rows,
        "cross_split_exact_duplicates": duplicates, "exact_cross_split_duplicate_groups": len(duplicates),
        "near_duplicate_or_sequence_leakage_checked": False,
        "note": "SHA-256 image bytes+label bytes+relative split/name; exact duplicate check only",
        "warnings": warning_messages,
    }
    if destination is not None:
        destination.mkdir(parents=True, exist_ok=True)
        _write_csv(destination / "dataset_audit.csv", split_rows,
                   ("split", "images", "labels", "boxes", "empty_labels"))
        _write_csv(destination / "class_distribution.csv", class_rows,
                   ("split", "class_id", "class_name", "boxes"))
        _write_csv(destination / "bbox_distribution.csv", box_rows,
                   ("split", "class_id", "class_name", "normalized_area", "normalized_width", "normalized_height"))
        duplicate_csv = [{**row, "occurrences": json.dumps(row["occurrences"], ensure_ascii=False)}
                         for row in duplicates]
        _write_csv(destination / "cross_split_exact_duplicates.csv", duplicate_csv, ("sha256", "occurrences"))
        (destination / "dataset_audit.json").write_text(
            json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8"
        )
        (destination / "dataset_fingerprint.json").write_text(
            json.dumps({key: report[key] for key in (
                "sha256", "source_path", "note", "near_duplicate_or_sequence_leakage_checked",
                "exact_cross_split_duplicate_groups",
            )}, indent=2, ensure_ascii=False), encoding="utf-8"
        )
    return report

def verify_dataset_unchanged(report: dict[str, Any]) -> None:
    """Periksa nama dan seluruh byte lagi, tanpa mengulang decode/validasi isi yang identik."""
    root = Path(report["source_path"]).resolve()
    actual_files = {}
    fingerprint = hashlib.sha256()
    for split in SPLITS:
        images = _by_stem(_files(root / split / "images", labels=False), "image")
        labels = _by_stem(_files(root / split / "labels", labels=True), "label")
        if images.keys() != labels.keys() or not images:
            raise DatasetAuditError(f"Dataset pairs changed after audit: {root / split}")
        for stem, image in sorted(images.items()):
            label = labels[stem]
            image_hash, raw = _sha256_file(image), label.read_bytes()
            actual_files[image.relative_to(root).as_posix()] = image_hash
            actual_files[label.relative_to(root).as_posix()] = hashlib.sha256(raw).hexdigest()
            fingerprint.update(f"{split}/{image.name}\n{image_hash}\n{label.name}\n".encode())
            fingerprint.update(raw)
    if actual_files != report["file_sha256"] or fingerprint.hexdigest() != report["fingerprint"]:
        raise DatasetAuditError("Dataset file names/bytes changed after audit; start a new audited run")

def write_runtime_yaml(root: str | Path, out_path: str | Path, *, audit: dict | None = None) -> Path:
    """Write a five-class data YAML outside the original dataset."""
    import yaml

    root = Path(root).expanduser().resolve()
    destination = _outside_source(Path(out_path), root)
    for split in SPLITS:
        if not (root / split / "images").is_dir() or not (root / split / "labels").is_dir():
            raise DatasetAuditError(f"Missing dataset split layout at {root / split}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    data = {"path": str(root), **{split: str(root / split / "images") for split in SPLITS},
            "nc": 5, "names": list(CLASS_NAMES)}
    if audit is not None:
        if Path(audit["source_path"]).resolve() != root:
            raise DatasetAuditError("Audit manifest belongs to a different dataset root")
        data["audit_file_sha256"] = audit["file_sha256"]
    destination.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return destination

def create_synthetic_dataset(root: str | Path, seed: int = 42) -> Path:
    """Create 20 unique 64px PNGs, five classes, train10/val5/test5.

    The destination must be new or empty to prevent overwriting real data.
    This fixture is deliberately small and has no research metric meaning.
    """
    from PIL import Image, ImageDraw

    root = Path(root).expanduser().resolve()
    if root.exists() and (not root.is_dir() or any(root.iterdir())):
        raise DatasetAuditError(f"Synthetic fixture destination must be new or empty: {root}")
    rng = random.Random(seed)
    index = 0
    for split, count in (("train", 10), ("val", 5), ("test", 5)):
        image_dir, label_dir = root / split / "images", root / split / "labels"
        image_dir.mkdir(parents=True)
        label_dir.mkdir(parents=True)
        for offset in range(count):
            image = Image.new("RGB", (64, 64), tuple(rng.randrange(24, 80) for _ in range(3)))
            draw = ImageDraw.Draw(image)
            draw.rectangle((16, 16, 47, 47), fill=tuple(rng.randrange(128, 256) for _ in range(3)))
            # Piksel penanda memastikan setiap gambar sintetis memiliki isi unik.
            image.putpixel((0, 0), (index, seed % 256, 255 - index))
            name = f"synthetic_{index:03d}"
            image.save(image_dir / f"{name}.png")
            (label_dir / f"{name}.txt").write_text(f"{offset % 5} 0.5 0.5 0.5 0.5\n", encoding="utf-8")
            index += 1
    return root

from ultralytics.data.dataset import YOLODataset

class SafeYOLODataset(YOLODataset):
    """Keep label caches in run output and reject any silent scan changes."""

    def __init__(self, *args: Any, audit_cache_dir: str | Path, **kwargs: Any) -> None:
        data = kwargs.get("data", {})
        if not data.get("path"):
            raise DatasetAuditError("SafeYOLODataset requires an explicit data['path']")
        self.audit_cache_dir = _outside_source(Path(audit_cache_dir), Path(data["path"]).resolve())
        self.source_root = Path(data["path"]).resolve()
        self.audited_files = data.get("audit_file_sha256")
        if kwargs.get("cache") == "disk":
            raise DatasetAuditError("Disk image caching would write into the source dataset; use cache=False")
        if kwargs.get("task", "detect") != "detect":
            raise DatasetAuditError("Stage A SafeYOLODataset supports bbox detection only")
        self.audit_cache_dir.mkdir(parents=True, exist_ok=True)
        super().__init__(*args, **kwargs)

    def _load_or_scan_cache(self, cache_path: Path, cache_hash: str) -> tuple[dict, bool]:
        expected: dict[str, list[tuple[float, ...]]] = {}
        content_digest = hashlib.sha256(str(Path(cache_path).resolve()).encode())
        for image_name, label_name in zip(self.im_files, self.label_files, strict=True):
            image_path, label_path = Path(image_name), Path(label_name)
            image_hash = _sha256_file(image_path)
            if self.audited_files is None:
                _verify_image(image_path)
            raw, rows = _label_rows(label_path)
            if self.audited_files is not None:
                # Lewati decode berulang hanya setelah byte cocok dengan audit run ini.
                for path, digest in ((image_path, image_hash), (label_path, hashlib.sha256(raw).hexdigest())):
                    relative = path.resolve().relative_to(self.source_root).as_posix()
                    if self.audited_files.get(relative) != digest:
                        raise DatasetAuditError(f"Dataset bytes changed since audit: {path}")
            expected[image_name] = rows
            content_digest.update(image_name.encode())
            content_digest.update(image_hash.encode())
            content_digest.update(raw)
        # Nama cache berdasarkan isi file; cache lama tidak lolos saat isi berubah.
        redirected = self.audit_cache_dir / f"{content_digest.hexdigest()}.cache"
        cache, exists = super()._load_or_scan_cache(redirected, cache_hash)
        found, missing, empty, corrupt, total = cache["results"]
        actual_labels = cache["labels"]
        actual_images = [label["im_file"] for label in actual_labels]
        expected_boxes = sum(map(len, expected.values()))
        actual_boxes = sum(len(label["cls"]) for label in actual_labels)
        if (missing or corrupt or total != len(expected) or found != len(expected)
                or len(actual_images) != len(expected) or set(actual_images) != set(expected)
                or expected_boxes != actual_boxes):
            raise DatasetAuditError(
                "Ultralytics scan changed audited images/boxes: "
                f"expected images={len(expected)}, boxes={expected_boxes}; "
                f"loaded images={len(actual_images)}, boxes={actual_boxes}, "
                f"missing={missing}, corrupt={corrupt}. Details: {cache.get('msgs', [])}"
            )
        if empty != sum(not rows for rows in expected.values()):
            raise DatasetAuditError("Ultralytics scan changed the audited empty-label count")
        for label in actual_labels:
            wanted = np.asarray(expected[label["im_file"]], dtype=np.float32).reshape(-1, 5)
            actual = np.concatenate((label["cls"], label["bboxes"]), axis=1)
            if not np.array_equal(actual, wanted) or len(label.get("segments", [])):
                raise DatasetAuditError(f"Ultralytics scan changed label values: {label['im_file']}")
        return cache, exists

"""Ekstensi kecil trainer/validator Ultralytics: cache di output dan protokol tetap.

Detector, optimizer, loss deteksi, dan mekanisme Native KD memakai Ultralytics.
"""

from copy import copy

from pathlib import Path

import torch

from ultralytics.models.yolo.detect import DetectionTrainer, DetectionValidator

from ultralytics.utils.torch_utils import unwrap_model

def safe_dataset(args, img_path, batch, data, *, mode, stride, cache_dir):
    # Loader harus menerima seluruh data yang diaudit; test tidak masuk evaluasi.
    if args.fraction != 1.0 or args.classes is not None or args.single_cls:
        raise ValueError("Stage A requires all audited images/classes; filtering is disabled")
    if Path(str(img_path)).resolve() == (Path(data["path"]) / "test" / "images").resolve():
        raise ValueError("Stage A never constructs a test-set dataloader")
    return SafeYOLODataset(
        img_path=img_path, imgsz=args.imgsz, batch_size=batch, augment=mode == "train",
        hyp=copy(args), rect=args.rect or mode == "val", cache=False,
        single_cls=False, stride=stride, pad=0.0 if mode == "train" else 0.5,
        prefix=f"{mode}: ", task="detect", classes=None, data=data, fraction=1.0,
        audit_cache_dir=cache_dir,
    )

class SafeDetectionValidator(DetectionValidator):
    def build_dataset(self, img_path, mode="val", batch=None):
        if self.args.split != "val":
            raise ValueError("Stage A allows validation split only")
        return safe_dataset(self.args, img_path, batch or self.args.batch, self.data,
                            mode="val", stride=self.stride, cache_dir=self.save_dir / "cache")

class SafeDetectionTrainer(DetectionTrainer):
    def run_callbacks(self, event: str):
        super().run_callbacks(event)
        if event == "on_train_epoch_start":
            # Pada v8.4.155, nilai 3 mematikan retry OOM yang bisa mengecilkan batch.
            self._oom_retries = 3

    def _build_train_pipeline(self):
        if getattr(self, "_oom_retries", 0):
            raise RuntimeError("OOM recovery is disabled; keep the configured batch and start a separate run")
        return super()._build_train_pipeline()

    def _handle_nan_recovery(self, epoch):
        values = [self.loss, self.fitness]
        if isinstance(self.loss_items, dict):
            values.extend(self.loss_items.values())
        for value in values:
            if value is not None and not bool(torch.isfinite(torch.as_tensor(value)).all()):
                raise RuntimeError(f"Nonfinite loss/fitness at epoch {epoch + 1}; automatic recovery is disabled")
        return False

    def build_dataset(self, img_path, mode="train", batch=None):
        model = unwrap_model(self.model) if self.model is not None else None
        stride = max(int(model.stride.max()) if model is not None else 0, 32)
        return safe_dataset(self.args, img_path, batch or self.batch_size, self.data,
                            mode=mode, stride=stride, cache_dir=self.save_dir / "cache")

    def get_validator(self):
        original = super().get_validator()
        return SafeDetectionValidator(dataloader=original.dataloader, save_dir=original.save_dir,
                                      args=original.args, _callbacks=original.callbacks)

In [ ]:
from copy import deepcopy

FULL_PROTOCOL = {'imgsz': 640,
 'batch': 16,
 'epochs': 100,
 'patience': 25,
 'optimizer': 'AdamW',
 'lr0': 0.001,
 'lrf': 0.01,
 'momentum': 0.937,
 'weight_decay': 0.0005,
 'warmup_epochs': 3.0,
 'warmup_momentum': 0.8,
 'warmup_bias_lr': 0.1,
 'amp': True,
 'workers': 2,
 'cache': False,
 'close_mosaic': 10,
 'hsv_h': 0.015,
 'hsv_s': 0.7,
 'hsv_v': 0.4,
 'degrees': 0.0,
 'translate': 0.1,
 'scale': 0.5,
 'shear': 0.0,
 'perspective': 0.0,
 'flipud': 0.0,
 'fliplr': 0.5,
 'mosaic': 1.0,
 'mixup': 0.0,
 'cutmix': 0.0,
 'copy_paste': 0.0,
 'deterministic': True,
 'compile': False,
 'save': True,
 'save_period': -1,
 'plots': True,
 'val': True,
 'nbs': 64,
 'fraction': 1.0,
 'cls_remap': True}

SMOKE_PROTOCOL = {'imgsz': 160,
 'batch': 2,
 'epochs': 1,
 'patience': 1,
 'workers': 0,
 'warmup_epochs': 0.0,
 'amp': False,
 'nbs': 2,
 'close_mosaic': 0,
 'plots': False,
 'hsv_h': 0.0,
 'hsv_s': 0.0,
 'hsv_v': 0.0,
 'translate': 0.0,
 'scale': 0.0,
 'fliplr': 0.0,
 'mosaic': 0.0}

EVALUATION_PROTOCOL = {'split': 'val',
 'nms': None,
 'conf': 0.001,
 'iou': 0.7,
 'max_det': 300,
 'augment': False,
 'quantize': None,
 'save_json': False,
 'pilot_latency': True,
 'warmup_calls': 5,
 'timed_calls': 30}

DATASET_PROTOCOL = {'name': 'InaTRC',
 'root': None,
 'expected_images': 3250,
 'names': ['Kelas-1 (Non-truk)',
           'Kelas-2 (Truk 2 sumbu)',
           'Kelas-3 (Truk 3 sumbu)',
           'Kelas-4 (Truk 4 sumbu)',
           'Kelas-5 (Truk >=5 sumbu)']}

KD_PROTOCOL = {'teacher_checkpoint': None, 'dis': 6.0, 'custom_status': 'stage_b_pilot_not_final'}

TRACKING_PROTOCOL = {'mode': 'disabled',
 'entity': None,
 'project': 'skripsi-inatrc-yolo26',
 'group': 'inatrc-kd',
 'upload_checkpoint_each_epoch': False}

from copy import deepcopy

from pathlib import Path

import os

import yaml

def validate_output_location(output_root: str | Path, dataset_root: str | Path | None) -> None:
    """Cegah output/cache ditulis ke dalam dataset asli."""
    if dataset_root is None:
        return
    output = Path(output_root).expanduser().resolve()
    source = Path(dataset_root).expanduser().resolve()
    if output == source or source in output.parents:
        raise ValueError("Output must be outside the source dataset root")

def validate_config(config: dict) -> None:
    validate_output_location(config["output_root"], config["dataset"]["root"])
    for name, choices in (("stage", ("teacher", "student")), ("student", ("s", "n")),
                          ("method", ("none", "native", "crosskd", "csakd")), ("mode", ("smoke", "full"))):
        if config[name] not in choices:
            raise ValueError(f"Invalid {name}={config[name]!r}; choose {choices}")
    if config["stage"] == "teacher" and config["method"] != "none":
        raise ValueError("Teacher training requires --method none")
    if config["dataset"]["names"] != list(CLASS_NAMES):
        raise ValueError("InaTRC class order must match the reference protocol")
    if config["tracking"]["mode"] not in ("disabled", "offline", "online"):
        raise ValueError("W&B mode must be disabled, offline, or online")
    if not isinstance(config["seed"], int) or isinstance(config["seed"], bool) or config["seed"] < 0:
        raise ValueError("Seed must be a nonnegative integer")
    training = config["training"]
    if training["batch"] < 1 or int(training["batch"]) != training["batch"]:
        raise ValueError("An explicit positive integer batch is required; AutoBatch is disabled")
    if training["epochs"] < 1 or training["imgsz"] % 32:
        raise ValueError("epochs must be positive and imgsz a multiple of 32")
    if config["evaluation"]["split"] != "val" or config["evaluation"]["nms"] is not None:
        raise ValueError("Stage A evaluates val only with the fixed nms=None baseline protocol")
    if config["mode"] == "full" and not config["dataset"]["root"]:
        raise ValueError("Full mode requires --dataset-root; it never uses synthetic data")


def make_config():
    """Bangun parameter efektif dari pilihan di cell pertama."""
    training = deepcopy(FULL_PROTOCOL)
    evaluation = deepcopy(EVALUATION_PROTOCOL)
    evaluation.update(coco_area=bool(EVALUATE_COCO_AREA), save_json=bool(EVALUATE_COCO_AREA))
    if MODE == "smoke":
        training.update(SMOKE_PROTOCOL)
        evaluation["pilot_latency"] = False
    training["seed"] = SEED
    dataset = deepcopy(DATASET_PROTOCOL)
    dataset["root"] = discover_dataset_root(DATASET_ROOT, INPUT_ROOT, MODE)
    dataset["expected_images"] = EXPECTED_IMAGES
    kd = deepcopy(KD_PROTOCOL)
    kd["teacher_checkpoint"] = str(Path(TEACHER_CKPT).expanduser().resolve()) if TEACHER_CKPT else None
    tracking = deepcopy(TRACKING_PROTOCOL)
    tracking.update(mode=WANDB_MODE, project=WANDB_PROJECT, entity=WANDB_ENTITY)
    scale = "m" if STAGE == "teacher" else STUDENT
    model = f"yolo26{scale}.{'yaml' if MODE == 'smoke' else 'pt'}"
    if MODEL_WEIGHTS:
        checkpoint = Path(MODEL_WEIGHTS).expanduser().resolve()
        if MODE != "full" or not checkpoint.is_file() or checkpoint.suffix != ".pt":
            raise ValueError("MODEL_WEIGHTS harus checkpoint pretrained COCO lokal untuk mode full.")
        model = str(checkpoint)
    config = {"stage": STAGE, "student": STUDENT, "method": METHOD, "mode": MODE,
              "seed": SEED, "device": str(DEVICE), "model": model, "output_root": str(OUTPUT_ROOT),
              "training": training, "evaluation": evaluation, "dataset": dataset, "kd": kd,
              "tracking": tracking, "create_results_zip": bool(CREATE_RESULTS_ZIP)}
    validate_config(config)
    if STAGE == "teacher" and SEED != 42:
        raise ValueError("Teacher penelitian tetap seed 42; seed student boleh diubah.")
    if METHOD != "none" and MODE == "full" and not TEACHER_CKPT:
        raise ValueError("KD full memerlukan TEACHER_CKPT: best.pt teacher YOLO26m InaTRC lima kelas.")
    return config

In [ ]:
CONFIG = make_config()
print("Dataset:", CONFIG["dataset"]["root"] or "sintetis (smoke)", flush=True)
print("Protokol:", CONFIG["model"], MODE, "seed", SEED, "batch", CONFIG["training"]["batch"], flush=True)
# Audit lengkap sekali; loader dan pemeriksaan akhir mencocokkan seluruh byte dengan audit itu.

## Model dan Teacher
Full dimulai dari pretrained COCO sesuai ukuran. Hash checkpoint dan state student awal dicatat. Teacher KD harus YOLO26m lima kelas dengan urutan kelas yang sama; teacher sintetis hanya diperbolehkan pada smoke.

In [ ]:
import hashlib

import importlib.metadata

import json

import platform

import sys

from pathlib import Path

from typing import Any

def _state_hash(model) -> str:
    fingerprint = hashlib.sha256()
    for name, value in model.state_dict().items():
        fingerprint.update(name.encode("utf-8"))
        fingerprint.update(value.detach().cpu().contiguous().numpy().tobytes())
    return fingerprint.hexdigest()

## Algoritme KD
Loss deteksi Ultralytics untuk teacher/baseline.

In [ ]:
def configure_kd(config):
    """Teacher dan baseline hanya memakai loss deteksi."""
    return {}

## Logging per Epoch
Selalu tampil dan tersimpan lokal: loss, KD, metrik val, LR, fitness, durasi, is_best, dan best_epoch. Kejadian penyimpanan best.pt menjadi acuan; fitness seri mengikuti checkpoint terbaru. Final validation tidak menambah epoch.

In [ ]:
"""Catat epoch/best checkpoint ke file lokal dan, bila aktif, satu run W&B."""

import math

import json

import os

import time

from pathlib import Path

import pandas as pd

def numeric(value, *, name: str) -> float | None:
    if value is None:
        return None
    result = float(value)
    if not math.isfinite(result):
        raise RuntimeError(f"Non-finite {name}: {result}")
    return result

class ExperimentTracker:
    def __init__(self, config: dict, run_dir: Path):
        self.config = config
        self.run_dir = run_dir
        self.run = None
        self.finished = False
        self.upload_seconds = 0.0

    def start(self, metadata: dict) -> None:
        if self.config["tracking"]["mode"] == "disabled":
            return
        import wandb
        if wandb.run is not None:
            raise RuntimeError("An existing W&B run is active; refusing a duplicate run")
        if self.config["tracking"]["mode"] == "online":
            # Runner meneruskan secret lewat environment, bukan melalui konfigurasi YAML.
            key = os.getenv("WANDB_API_KEY")
            if not key:
                raise RuntimeError("W&B online requires WANDB_API_KEY from environment or Kaggle Secrets")
            wandb.login(key=key, relogin=False)
        self.run = wandb.init(
            mode=self.config["tracking"]["mode"], entity=self.config["tracking"]["entity"],
            project=self.config["tracking"]["project"], group=self.config["tracking"]["group"],
            name=self.run_dir.name, job_type=f"{self.config['stage']}-{self.config['method']}",
            tags=["InaTRC", self.config["mode"], f"seed-{self.config['seed']}"],
            dir=str(self.run_dir), config={**self.config, "metadata": metadata}, reinit=False,
        )
        self.run.summary["research/status"] = "running"
        self.run.summary["research/test_evaluated"] = False
        # Semua kurva memakai epoch sebagai sumbu, bukan jumlah panggilan callback.
        self.run.define_metric("epoch")
        self.run.define_metric("*", step_metric="epoch")

    def log(self, values: dict, step: int | None = None) -> None:
        if self.run is not None:
            self.run.log(values, step=step)

    def epoch_artifact(self, checkpoint: Path, epoch: int, is_best: bool) -> None:
        if self.run is None or not self.config["tracking"]["upload_checkpoint_each_epoch"]:
            return
        import wandb
        started = time.perf_counter()
        artifact = wandb.Artifact(f"epochs-{self.run.id}", type="model", metadata={"epoch": epoch})
        artifact.add_file(str(checkpoint), name="checkpoint.pt")
        logged = self.run.log_artifact(artifact, aliases=[f"epoch-{epoch:03d}"] + (["best"] if is_best else []))
        if self.config["tracking"]["mode"] == "online":
            logged.wait()
        self.upload_seconds += time.perf_counter() - started

    def finish(self, status: str, metrics: dict | None = None, error: str | None = None) -> None:
        if self.run is None or self.finished:
            return
        import wandb
        exit_status = status
        upload_started = None
        try:
            self.run.summary["research/status"] = status
            self.run.summary["research/test_evaluated"] = False
            training_summary = {}
            summary_path = self.run_dir / "training_summary.json"
            if summary_path.is_file():
                training_summary = json.loads(summary_path.read_text(encoding="utf-8"))
                self.run.summary["checkpoint/best_epoch"] = training_summary["best_epoch"]
                self.run.summary["fit/best_fitness"] = training_summary["best_fitness"]
                self.run.summary["fit/completed_epochs"] = training_summary["completed_epochs"]
            if error:
                key = os.getenv("WANDB_API_KEY")
                self.run.summary["research/error"] = error.replace(key, "[REDACTED]") if key else error
            if metrics is not None:
                for key in ("mAP50_95", "mAP50", "precision", "recall"):
                    self.run.summary[f"validation/{key}"] = metrics[key]
                self.run.summary["checkpoint/best_pt"] = str(self.run_dir / "weights" / "best.pt")
            if status == "completed":
                upload_started = time.perf_counter()
                artifact = wandb.Artifact(f"final-{self.run.id}", type="model",
                                          metadata={"mode": self.config["mode"], "test_evaluated": False,
                                                    **training_summary})
                for name in ("weights/best.pt", "weights/last.pt", "metrics.json", "per_class.csv", "results.csv",
                             "epoch_history.csv", "checkpoint_manifest.csv", "training_summary.json",
                             "checkpoint_audit.json", "complexity.json", "resolved_config.yaml",
                             "coco_area.json", "coco_metrics.csv", "metrics.csv", "results.png",
                             "metadata.json", "run_summary.json"):
                    path = self.run_dir / name
                    if path.is_file():
                        artifact.add_file(str(path), name=name)
                for folder in ("reports", "validation"):
                    for path in sorted((self.run_dir / folder).rglob("*")):
                        if path.is_file() and "cache" not in path.relative_to(self.run_dir).parts:
                            artifact.add_file(str(path), name=path.relative_to(self.run_dir).as_posix())
                logged = self.run.log_artifact(artifact, aliases=["best-checkpoint", "final"])
                if self.config["tracking"]["mode"] == "online":
                    logged.wait()
        except BaseException as exc:
            exit_status = "failed"
            self.run.summary["research/status"] = "failed"
            error_text = f"{type(exc).__name__}: {exc}"
            key = os.getenv("WANDB_API_KEY")
            self.run.summary["research/error"] = error_text.replace(key, "[REDACTED]") if key else error_text
            if key and key in error_text:
                raise RuntimeError(self.run.summary["research/error"]) from None
            raise
        finally:
            if upload_started is not None:
                self.upload_seconds += time.perf_counter() - upload_started
            self.finished = True
            self.run.finish(exit_code=0 if exit_status == "completed" else 1)

class EpochRecorder:
    """Catat epoch dan best epoch meski W&B dan checkpoint tiap epoch dimatikan."""
    def __init__(self, config: dict, run_dir: Path, tracker: ExperimentTracker):
        self.config, self.run_dir, self.tracker = config, run_dir, tracker
        self.last_saved_epoch = 0
        self.last_logged_epoch = 0
        self.best_epoch = None
        self.best_fitness = None
        self.epoch_started = None
        self.rows: list[dict] = []
        self.checkpoints: list[dict] = []

    def on_epoch_start(self, trainer) -> None:
        self.epoch_started = time.perf_counter()

    def on_save(self, trainer) -> None:
        epoch = int(trainer.epoch + 1)
        fitness = numeric(trainer.fitness, name="fitness")
        best = numeric(trainer.best_fitness, name="best_fitness")
        is_best = fitness is not None and best is not None and fitness == best
        if is_best:
            # Ultralytics menimpa best.pt juga saat fitness seri; epoch terbaru berlaku.
            self.best_epoch, self.best_fitness = epoch, best
        self.last_saved_epoch = epoch
        self.checkpoints.append({"epoch": epoch, "fitness": fitness, "best_fitness": best,
                                 "is_best_at_save": is_best, "best_epoch": self.best_epoch})
        pd.DataFrame(self.checkpoints).to_csv(self.run_dir / "checkpoint_manifest.csv", index=False)
        if self.config["tracking"]["upload_checkpoint_each_epoch"]:
            checkpoint = Path(trainer.wdir) / f"epoch{trainer.epoch}.pt"
            if not checkpoint.is_file():
                raise RuntimeError(f"Missing requested epoch checkpoint: {checkpoint}")
            self.tracker.epoch_artifact(checkpoint, epoch, is_best)

    def on_fit_epoch_end(self, trainer) -> None:
        epoch = int(trainer.epoch + 1)
        if epoch != self.last_saved_epoch or epoch <= self.last_logged_epoch:
            return  # Validasi final memanggil callback lagi; epoch cukup dicatat sekali.
        values = {"epoch": epoch}
        for group in (trainer.label_loss_items(trainer.tloss, prefix="train"), trainer.metrics or {}, trainer.lr or {}):
            for key, value in group.items():
                value = numeric(value, name=key)
                if value is not None:
                    values[key] = value
        values["fit/fitness"] = numeric(trainer.fitness, name="fitness")
        values["fit/best_fitness"] = numeric(trainer.best_fitness, name="best_fitness")
        values["checkpoint/best_epoch"] = self.best_epoch
        values["checkpoint/is_best"] = self.checkpoints[-1]["is_best_at_save"]
        if self.epoch_started is not None:
            values["fit/epoch_seconds"] = time.perf_counter() - self.epoch_started
        import torch
        if trainer.device.type == "cuda":
            values["system/gpu_memory_allocated_gib"] = torch.cuda.memory_allocated(trainer.device) / 2**30
            values["system/gpu_memory_reserved_gib"] = torch.cuda.memory_reserved(trainer.device) / 2**30
        box = getattr(getattr(getattr(trainer, "validator", None), "metrics", None), "box", None)
        if box is not None:
            for position, class_id in enumerate(box.ap_class_index):
                for metric, array in (("precision", box.p), ("recall", box.r), ("AP50", box.ap50), ("AP50_95", box.ap)):
                    values[f"per_class/{int(class_id)}/{metric}"] = numeric(array[position], name=metric)
        self.rows.append(values)
        pd.DataFrame(self.rows).to_csv(self.run_dir / "epoch_history.csv", index=False)
        self.tracker.log(values, step=epoch)
        self.last_logged_epoch = epoch
        map_value = values.get("metrics/mAP50-95(B)")
        map_text = "belum tersedia" if map_value is None else f"{map_value:.5f}"
        updated = "ya" if values["checkpoint/is_best"] else "tidak"
        print(f"Epoch {epoch}/{self.config['training']['epochs']} | mAP50-95 {map_text} | "
              f"Best epoch {self.best_epoch} | best.pt diperbarui: {updated}", flush=True)

    def attach(self, model) -> None:
        model.add_callback("on_train_epoch_start", self.on_epoch_start)
        model.add_callback("on_model_save", self.on_save)
        model.add_callback("on_fit_epoch_end", self.on_fit_epoch_end)

    def verify(self, results_csv: Path) -> dict:
        history = pd.read_csv(results_csv)
        history.columns = history.columns.str.strip()
        if len(history) != len(self.rows) or len(history) != self.last_saved_epoch:
            raise RuntimeError("CSV, epoch logger, and checkpoint events disagree")
        column = "metrics/mAP50-95(B)"
        if column not in history or not all(math.isfinite(float(v)) for v in history[column]):
            raise RuntimeError("Training history lacks finite mAP50-95")
        if self.config["method"] == "none" and any("dis_loss" in col for col in history):
            raise RuntimeError("Baseline unexpectedly contains KD loss")
        if self.config["method"] != "none" and not any("dis_loss" in col for col in history):
            raise RuntimeError("KD history lacks dis_loss; refusing a silent baseline")
        for method, components in (("crosskd", ("kd_cls_loss", "kd_box_loss")),
                                   ("csakd", ("kd_agka_loss", "kd_cross_loss"))):
            if self.config["method"] == method and not all(any(name in column for column in history) for name in components):
                raise RuntimeError(f"{method} history lacks algorithm-specific loss components")
        if self.best_epoch is None:
            raise RuntimeError("No best checkpoint epoch was recorded")
        summary = {"completed_epochs": len(history), "best_epoch": self.best_epoch,
                   "best_fitness": self.best_fitness,
                   "early_stopped": len(history) < self.config["training"]["epochs"]}
        write_json(self.run_dir / "training_summary.json", summary)
        return summary

## Evaluasi best.pt
Muat ulang YOLO(best.pt), val FP32: nms=None, conf=0.001, IoU=0.7, max_det=300, augmentasi off. Parameter/FLOPs dari checkpoint lima kelas sebelum fusion; latency, VRAM, dan durasi tetap dicatat. COCO area opsional (EVALUATE_COCO_AREA=True) memakai bbox pixel asli/maxDets=[1,10,100]; definisi metrik utama tetap sama.

In [ ]:
"""COCO AP berdasarkan luas bbox asli; metrik utama Ultralytics tetap terpisah."""

from copy import deepcopy

from importlib.metadata import PackageNotFoundError, version

import math

from numbers import Integral

from pathlib import Path

import numpy as np

COCO_EVAL_VERSION = "1.6.7"

COCO_AREA_RANGES = [[0, 1e10], [0, 32 ** 2], [32 ** 2, 96 ** 2], [96 ** 2, 1e10]]

COCO_AREA_LABELS = ["all", "small", "medium", "large"]

COCO_METRIC_KEYS = ("AP", "AP50", "AP_S", "AP_M", "AP_L")

def _coco_integer(value, context):
    if isinstance(value, bool) or not isinstance(value, Integral):
        raise ValueError(f"COCO {context} must be an integer: {value!r}")
    return int(value)

def _coco_bbox(value, context):
    if not isinstance(value, (list, tuple)) or len(value) != 4:
        raise ValueError(f"COCO {context} requires four bbox coordinates")
    if any(isinstance(number, bool) or not isinstance(number, (int, float)) for number in value):
        raise ValueError(f"COCO {context} bbox must contain finite numbers")
    bbox = [float(number) for number in value]
    if not all(math.isfinite(number) for number in bbox) or min(bbox[2:]) <= 0:
        raise ValueError(f"COCO {context} bbox must be finite with positive width/height")
    return bbox

def remap_coco_predictions(ground_truth, predictions, pred_counts):
    """Cocokkan urutan ekspor per gambar, termasuk nama file yang bukan angka."""
    images = ground_truth["images"]
    if len(pred_counts) != len(images):
        raise ValueError("COCO prediction-count length differs from the ground-truth image count")
    counts = [_coco_integer(count, "prediction count") for count in pred_counts]
    if any(count < 0 for count in counts) or sum(counts) != len(predictions):
        raise ValueError("COCO prediction counts do not match the exported detections")

    remapped, offset, seen_sources = [], 0, set()
    for image, count in zip(images, counts, strict=True):
        chunk = predictions[offset:offset + count]
        offset += count
        sources = {(row.get("file_name"), row.get("image_id")) for row in chunk}
        if len(sources) > 1:
            raise ValueError(f"COCO detections from different images were mixed for image {image['id']}")
        if sources:
            source = next(iter(sources))
            if source in seen_sources:
                raise ValueError("COCO detections repeat the same source image in different image groups")
            seen_sources.add(source)
            if image.get("file_name"):
                filename = Path(image["file_name"]).name
                stem = Path(filename).stem
                expected_id = int(stem) if stem.isnumeric() else stem
                if source != (filename, expected_id):
                    raise ValueError(f"COCO prediction source does not match ground-truth image {filename}")
        remapped.extend({**row, "image_id": image["id"]} for row in chunk)
    return remapped

def _validate_coco_inputs(ground_truth, predictions, pred_counts):
    if not isinstance(ground_truth, dict):
        raise ValueError("COCO ground truth must be a dictionary")
    for key in ("images", "annotations", "categories"):
        if not isinstance(ground_truth.get(key), list):
            raise ValueError(f"COCO ground truth requires a {key} list")
    if not isinstance(predictions, list):
        raise ValueError("COCO predictions must be a list")
    if not isinstance(pred_counts, (list, tuple)):
        raise ValueError("COCO prediction counts must be a list or tuple")

    ground_truth = deepcopy(ground_truth)
    image_ids, category_ids = set(), set()
    for key, target in (("images", image_ids), ("categories", category_ids)):
        for item in ground_truth[key]:
            item["id"] = _coco_integer(item["id"], f"{key} ID")
            if item["id"] in target:
                raise ValueError(f"COCO contains duplicate {key} IDs")
            target.add(item["id"])
    if not category_ids:
        raise ValueError("COCO ground truth requires at least one category")
    annotation_ids = set()
    for row in ground_truth["annotations"]:
        row["id"] = _coco_integer(row["id"], "annotation ID")
        row["image_id"] = _coco_integer(row["image_id"], "annotation image ID")
        row["category_id"] = _coco_integer(row["category_id"], "annotation category ID")
        if row["id"] in annotation_ids:
            raise ValueError("COCO contains duplicate annotation IDs")
        annotation_ids.add(row["id"])
        if row["image_id"] not in image_ids or row["category_id"] not in category_ids:
            raise ValueError("COCO annotation refers to an unknown image/category")
        row["bbox"] = _coco_bbox(row["bbox"], "annotation")
        area = float(row["area"])
        if not math.isfinite(area) or not math.isclose(area, row["bbox"][2] * row["bbox"][3], rel_tol=1e-6):
            raise ValueError("COCO annotation area must equal the original-pixel bbox area")
        row["area"] = area
        row["iscrowd"] = _coco_integer(row.get("iscrowd", 0), "iscrowd")
        if row["iscrowd"] not in (0, 1):
            raise ValueError("COCO iscrowd must be zero or one")

    remapped = remap_coco_predictions(ground_truth, predictions, pred_counts)
    for row in remapped:
        row["category_id"] = _coco_integer(row["category_id"], "prediction category ID")
        if row["category_id"] not in category_ids:
            raise ValueError("COCO prediction refers to an unknown category")
        row["bbox"] = _coco_bbox(row["bbox"], "prediction")
        row["score"] = float(row["score"])
        if not math.isfinite(row["score"]) or not 0 <= row["score"] <= 1:
            raise ValueError("COCO prediction score must be finite and between zero and one")
    return ground_truth, remapped

def _coco_mean_or_null(values):
    values = np.asarray(values, dtype=float)
    valid = values[np.isfinite(values) & (values >= 0)]
    return float(valid.mean()) if valid.size else None

def evaluate_coco_area(ground_truth, predictions, pred_counts):
    """Hitung AP COCO terpisah; kelompok tanpa ground truth mempunyai nilai null."""
    try:
        installed = version("faster-coco-eval")
    except PackageNotFoundError:
        raise RuntimeError(f"COCO evaluation requires faster-coco-eval=={COCO_EVAL_VERSION}") from None
    if installed != COCO_EVAL_VERSION:
        raise RuntimeError(f"COCO evaluation requires faster-coco-eval=={COCO_EVAL_VERSION}; found {installed}")
    from faster_coco_eval import COCO, COCOeval_faster

    ground_truth, predictions = _validate_coco_inputs(ground_truth, predictions, pred_counts)
    annotation_api = COCO(ground_truth, use_deepcopy=True)
    if predictions:
        prediction_api = annotation_api.loadRes(predictions)
    else:
        # loadRes([]) pada versi pin mengakses elemen pertama; COCO kosong tetap sah.
        prediction_api = COCO({"images": deepcopy(ground_truth["images"]),
                               "categories": deepcopy(ground_truth["categories"]), "annotations": []})
    evaluator = COCOeval_faster(annotation_api, prediction_api, iouType="bbox")
    evaluator.params.imgIds = [image["id"] for image in ground_truth["images"]]
    evaluator.params.catIds = sorted(category["id"] for category in ground_truth["categories"])
    evaluator.params.iouThrs = np.linspace(0.5, 0.95, 10)
    evaluator.params.maxDets = [1, 10, 100]
    evaluator.params.areaRng = deepcopy(COCO_AREA_RANGES)
    evaluator.params.areaRngLbl = list(COCO_AREA_LABELS)
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    # precision: IoU, recall, kelas, kelompok luas, maxDets; -1 berarti tidak valid.
    precision = np.asarray(evaluator.eval["precision"])
    slices = (precision[:, :, :, 0, -1], precision[0, :, :, 0, -1],
              precision[:, :, :, 1, -1], precision[:, :, :, 2, -1], precision[:, :, :, 3, -1])
    report = {key: _coco_mean_or_null(values) for key, values in zip(COCO_METRIC_KEYS, slices, strict=True)}
    report["per_class"] = []
    names = {category["id"]: category.get("name") for category in ground_truth["categories"]}
    for position, category_id in enumerate(evaluator.params.catIds):
        class_slices = (precision[:, :, position, 0, -1], precision[0, :, position, 0, -1],
                        precision[:, :, position, 1, -1], precision[:, :, position, 2, -1],
                        precision[:, :, position, 3, -1])
        report["per_class"].append({"category_id": int(category_id), "class_name": names[category_id],
                                   **{key: _coco_mean_or_null(values) for key, values in
                                      zip(COCO_METRIC_KEYS, class_slices, strict=True)}})
    report.update(image_count=len(ground_truth["images"]), ground_truth_count=len(ground_truth["annotations"]),
                  prediction_count=len(predictions))
    report["protocol"] = {"evaluator": f"faster-coco-eval=={COCO_EVAL_VERSION}", "iou_type": "bbox",
                          "iou_thresholds": [round(float(value), 2) for value in evaluator.params.iouThrs],
                          "max_detections": [1, 10, 100], "area_ranges_pixels_squared": deepcopy(COCO_AREA_RANGES),
                          "area_labels": list(COCO_AREA_LABELS), "primary_metric": "Ultralytics mAP50_95 unchanged"}
    return report

class AreaAuditValidator(SafeDetectionValidator):
    """Validator final dengan laporan COCO tambahan, tanpa mengganti fitness native."""

    def init_metrics(self, model):
        self.gdict = None
        super().init_metrics(model)
        if self.gdict is not None:
            for class_id, category in enumerate(self.gdict["categories"]):
                category["name"] = self.names[class_id]
        self._coco_area_evaluated = False
        self.metrics.coco_area = None

    def update_metrics(self, preds, batch):
        image_offset = len(self.gdict["images"]) if self.build_gdict else 0
        super().update_metrics(preds, batch)
        if self.build_gdict:
            added = self.gdict["images"][image_offset:]
            if len(added) != len(batch["im_file"]):
                raise RuntimeError("COCO ground-truth export lost validation images")
            for image, filename, shape in zip(added, batch["im_file"], batch["ori_shape"], strict=True):
                image.update(file_name=Path(filename).name, height=int(shape[0]), width=int(shape[1]))

    def get_stats(self):
        stats = super().get_stats()
        # Placeholder nol upstream bukan laporan COCO; hindari dua versi metrik luas.
        for size in ("small", "medium", "large"):
            stats.pop(f"metrics/mAP_{size}(B)", None)
        if not self.training and self.args.save_json:
            self.eval_json(stats)  # Tetap jalan ketika seluruh prediksi kosong.
        return stats

    def eval_json(self, stats):
        if self.training or not self.args.save_json or getattr(self, "_coco_area_evaluated", False):
            return stats
        if not self.gdict or not self.is_custom_json:
            raise RuntimeError("COCO area audit requires the custom InaTRC validation export")
        try:
            self.metrics.coco_area = evaluate_coco_area(self.gdict, self.jdict, self.pred_counts)
        except Exception as error:
            raise RuntimeError(f"COCO area evaluation failed: {error}") from error
        self._coco_area_evaluated = True
        return stats

"""Muat ulang best.pt dan hitung metrik pada val dengan protokol yang tetap."""

import math

import time

from copy import deepcopy

from pathlib import Path

from statistics import median

import numpy as np

import pandas as pd

import torch

from ultralytics import YOLO

from ultralytics.utils.torch_utils import get_flops

def evaluate_checkpoint(checkpoint: str | Path, data_yaml: str | Path, run_dir: str | Path, config: dict) -> dict:
    checkpoint, run_dir = Path(checkpoint), Path(run_dir)
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Missing best checkpoint: {checkpoint}")
    evaluation, training = config["evaluation"], config["training"]
    if evaluation["split"] != "val" or evaluation["nms"] is not None:
        raise ValueError("Only the fixed val/nms=None protocol is available in Stage A")
    # Evaluasi model tersimpan secara mandiri; teacher/adapter KD tidak diperlukan.
    model = YOLO(str(checkpoint))
    native = model.model
    head = native.model[-1]
    if int(head.nc) != 5 or [native.names[i] for i in range(5)] != config["dataset"]["names"]:
        raise RuntimeError("Checkpoint class names/head do not match the InaTRC protocol")
    if hasattr(native, "teacher_model") or hasattr(native, "projector") or hasattr(native, "student_model"):
        raise RuntimeError("Checkpoint still contains the KD training wrapper")
    audit = {"path": str(checkpoint.resolve()), "sha256": sha256(checkpoint),
             "size_mb": checkpoint.stat().st_size / 1e6,
             "parameters": sum(parameter.numel() for parameter in native.parameters()),
             "nc": int(head.nc), "teacher_present": False, "projector_present": False}
    # Salinan sebelum fusion menjaga penghitungan FLOPs memakai model checkpoint asli.
    complexity_probe = deepcopy(native).float().eval()
    result = model.val(
        validator=AreaAuditValidator if evaluation["coco_area"] else SafeDetectionValidator, data=str(data_yaml), split="val",
        imgsz=training["imgsz"], batch=training["batch"], workers=training["workers"],
        device=config["device"], plots=training["plots"], save_json=evaluation["coco_area"],
        nms=None, conf=evaluation["conf"], iou=evaluation["iou"], max_det=evaluation["max_det"],
        augment=False, quantize=evaluation["quantize"],
        project=str(run_dir / "validation"), name="best-val", exist_ok=False,
    )
    box = result.box
    metrics = {"split": "val", "test_evaluated": False, "mode": config["mode"],
               "nonfinal": config["mode"] == "smoke", "nms": None,
               "precision": float(box.mp), "recall": float(box.mr),
               "mAP50": float(box.map50), "mAP50_95": float(box.map),
               "checkpoint": audit, "validation_dir": str(result.save_dir),
               "validation_precision": "float32" if evaluation["quantize"] is None else evaluation["quantize"]}
    for key in ("precision", "recall", "mAP50", "mAP50_95"):
        if not math.isfinite(metrics[key]):
            raise RuntimeError(f"Non-finite validation metric {key}")
    indices = {int(class_id): position for position, class_id in enumerate(box.ap_class_index)}
    per_class = []
    ap = np.asarray(box.all_ap)
    for class_id, name in enumerate(config["dataset"]["names"]):
        position = indices.get(class_id)
        # Kelas tanpa AP valid tetap null, agar tidak dianggap memperoleh nilai nol.
        row = {"class_id": class_id, "class_name": name,
               **dict.fromkeys(("precision", "recall", "AP50", "AP75", "AP50_95"))}
        if position is not None:
            values = (box.p[position], box.r[position], box.ap50[position], ap[position, 5], ap[position].mean())
            for key, value in zip(("precision", "recall", "AP50", "AP75", "AP50_95"), values, strict=True):
                row[key] = float(value)
                if not math.isfinite(row[key]):
                    raise RuntimeError(f"Non-finite {key} for class {class_id}")
        per_class.append(row)
    metrics["per_class"] = per_class
    if evaluation["coco_area"]:
        if getattr(result, "coco_area", None) is None:
            raise RuntimeError("COCO area evaluator did not return final metrics")
        metrics["coco_area"] = result.coco_area
        write_json(run_dir / "coco_area.json", result.coco_area)
        pd.DataFrame([{key: result.coco_area[key] for key in COCO_METRIC_KEYS}]).to_csv(run_dir / "coco_metrics.csv", index=False)
    complexity = {"parameters": audit["parameters"], "imgsz": training["imgsz"],
                  "GFLOPs_estimate": None,
                  "convention": "2 * THOP MACs / 1e9, float32 unfused checkpoint; estimate"}
    gflops = float(get_flops(complexity_probe, imgsz=training["imgsz"]))
    if math.isfinite(gflops) and gflops > 0:
        complexity["GFLOPs_estimate"] = gflops
    else:
        complexity["profile_status"] = "unavailable_from_upstream_profiler"
    del complexity_probe
    metrics["complexity"] = complexity
    metrics["speed_ms_per_image"] = {key: float(value) for key, value in result.speed.items()}
    if evaluation["pilot_latency"]:
        import yaml
        data = yaml.safe_load(Path(data_yaml).read_text(encoding="utf-8"))
        image = next(iter(sorted(path for path in Path(data["val"]).iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)))
        times = []
        for index in range(evaluation["warmup_calls"] + evaluation["timed_calls"]):
            if str(config["device"]) != "cpu":
                torch.cuda.synchronize()
            start = time.perf_counter()
            model.predict(str(image), imgsz=training["imgsz"], device=config["device"],
                          nms=None, verbose=False, save=False)
            if str(config["device"]) != "cpu":
                torch.cuda.synchronize()
            if index >= evaluation["warmup_calls"]:
                times.append((time.perf_counter() - start) * 1000)
        metrics["pilot_latency"] = {"batch": 1, "imgsz": training["imgsz"],
                                    "median_wall_ms": median(times), "mean_wall_ms": float(np.mean(times)),
                                    "p95_wall_ms": float(np.percentile(times, 95)),
                                    "note": "Single val image; includes I/O, Python, preprocessing and postprocessing; hardware-specific estimate"}
    pd.DataFrame(per_class).to_csv(run_dir / "per_class.csv", index=False)
    write_json(run_dir / "checkpoint_audit.json", audit)
    write_json(run_dir / "complexity.json", complexity)
    pd.DataFrame([{key: metrics[key] for key in ("precision", "recall", "mAP50", "mAP50_95")}]).to_csv(run_dir / "metrics.csv", index=False)
    write_json(run_dir / "metrics.json", metrics)
    return metrics

## Training
Satu run memakai folder unik. OOM/nonfinite menghentikan run. best.pt dan last.pt disimpan bila tersedia; finalisasi Ultralytics dapat menghapus optimizer dari checkpoint.

In [ ]:
NOTEBOOK_PROVENANCE = {'format': 'inline-notebook',
 'preset': '01_teacher_yolo26m',
 'code_sha256': 'd3bf84f3507b1b9c0811b7e8c3c1b22e043ee42f98edcd1eea8f4cb7a90c846e',
 'code_hash_scheme': 'sha256_joined_code_cells_with_hash_placeholder',
 'ultralytics': '8.4.155'}

def source_provenance():
    return deepcopy(NOTEBOOK_PROVENANCE)

"""Alur utama: cek environment/data → training → muat best.pt → evaluasi → simpan hasil."""

import json

import os

import time

import uuid

from datetime import datetime, timezone

from pathlib import Path

import torch

import yaml

from ultralytics import YOLO, settings

from ultralytics.cfg import get_cfg

from ultralytics.utils.torch_utils import init_seeds

from ultralytics.utils.torch_utils import unwrap_model

def run_experiment(config: dict) -> dict:
    validate_config(config)
    # Logging W&B hanya melalui tracker manual, agar tidak terbentuk dua run.
    settings.update({"wandb": False})
    output = Path(config["output_root"])
    scale = "m" if config["stage"] == "teacher" else config["student"]
    name = (f"{config['stage']}-yolo26{scale}-{config['method']}-seed{config['seed']}-{config['mode']}-"
            f"{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid.uuid4().hex[:8]}")
    run_dir = output / name
    run_dir.mkdir(parents=True, exist_ok=False)
    (run_dir / "reports").mkdir()
    started = time.perf_counter()
    tracker = ExperimentTracker(config, run_dir)
    metadata = {"status": "preflight", "nonfinal": config["mode"] == "smoke",
                "test_evaluated": False, "source": source_provenance(),
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "stage": config["stage"], "student": config["student"], "method": config["method"],
                "seed": config["seed"], "mode": config["mode"], "run_dir": str(run_dir)}
    try:
        environment = ENVIRONMENT
        if environment["requested_device"] != config["device"]:
            raise ValueError("Device berubah setelah preflight; jalankan ulang cell Pemeriksaan GPU.")
        metadata["environment"] = environment
        if config["device"] == "cpu":
            torch.set_num_threads(min(4, torch.get_num_threads()))
        if config["dataset"]["root"]:
            actual_audit = audit_dataset(config["dataset"]["root"], run_dir / "reports" / "inatrc",
                                         expected_images=config["dataset"]["expected_images"])
            metadata["inatrc_dataset_fingerprint"] = actual_audit["fingerprint"]
        # Smoke selalu memakai data buatan; dataset asli yang terpasang tetap diaudit.
        if config["mode"] == "smoke":
            data_root = create_synthetic_dataset(run_dir / "synthetic_data", seed=config["seed"])
            audit = audit_dataset(data_root, run_dir / "reports" / "training_data", expected_images=20)
            config["initialization"] = "random_yaml_diagnostics_only"
        else:
            data_root = Path(config["dataset"]["root"])
            audit = actual_audit
            config["initialization"] = "pretrained"
        config["training_dataset_root"] = str(data_root)
        config["dataset_fingerprint"] = audit["fingerprint"]
        data_yaml = write_runtime_yaml(data_root, run_dir / "data_runtime.yaml", audit=audit)
        init_seeds(config["seed"], deterministic=config["training"]["deterministic"])
        kd_args = configure_kd(config)
        model = YOLO(config["model"])
        if config["mode"] == "full":
            yaml_file = Path(str(model.model.yaml.get("yaml_file", ""))).stem
            if yaml_file not in ("yolo26", f"yolo26{scale}") or model.model.yaml.get("scale") != scale:
                raise ValueError(f"Initial pretrained weights must match YOLO26{scale}")
            if int(model.model.model[-1].nc) != 80:
                raise ValueError("Research initialization must use COCO pretrained weights, not a fine-tuned baseline checkpoint")
            checkpoint = Path(model.ckpt_path)
            metadata["initial_checkpoint"] = {"path": str(checkpoint.resolve()), "sha256": sha256(checkpoint)}
        if kd_args:
            metadata["teacher_checkpoint_sha256"] = sha256(kd_args["distill_model"])
            if TEACHER_SHA256 and metadata["teacher_checkpoint_sha256"] != TEACHER_SHA256:
                raise ValueError("Hash teacher berbeda dari TEACHER_SHA256 yang ditentukan.")
        train_args = {**config["training"], **kd_args, "data": str(data_yaml),
                      "device": config["device"], "project": str(output), "name": name, "exist_ok": True,
                      "pretrained": config["mode"] == "full", "nms": None,
                      "conf": config["evaluation"]["conf"], "iou": config["evaluation"]["iou"],
                      "max_det": config["evaluation"]["max_det"], "augment": False}
        config["ultralytics_effective_config"] = vars(get_cfg(overrides=train_args))
        # Simpan parameter yang benar-benar dipakai, termasuk default dari Ultralytics.
        (run_dir / "resolved_config.yaml").write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
        write_json(run_dir / "metadata.json", metadata)
        tracker.start(metadata)
        recorder = EpochRecorder(config, run_dir, tracker)
        recorder.attach(model)

        def record_runtime_initialization(trainer):
            # Hash awal membantu membandingkan inisialisasi antar eksperimen.
            runtime = unwrap_model(trainer.model)
            student = getattr(runtime, "student_model", runtime)
            metadata["initial_student_state_sha256"] = _state_hash(student)
            if hasattr(runtime, "teacher_model"):
                metadata["initial_teacher_state_sha256"] = _state_hash(runtime.teacher_model)
                optimized = {id(p) for group in trainer.optimizer.param_groups for p in group["params"]}
                if not all(id(p) in optimized for p in runtime.projector.parameters()):
                    raise RuntimeError("KD adapter missing from optimizer")
                metadata["adapter_parameters"] = sum(p.numel() for p in runtime.projector.parameters())

        model.add_callback("on_pretrain_routine_end", record_runtime_initialization)
        if config["device"] == "cpu":
            torch.set_num_threads(min(4, torch.get_num_threads()))
        else:
            torch.cuda.reset_peak_memory_stats()
        training_started = time.perf_counter()
        # Teacher/baseline/Native berbagi trainer; custom KD hanya mengganti bagian KD.
        trainer = SafeDetectionTrainer
        model.train(trainer=trainer, **train_args)
        metadata["training_seconds"] = time.perf_counter() - training_started
        metadata["amp_requested"] = config["training"]["amp"]
        metadata["amp_enabled"] = bool(model.trainer.amp)
        metadata["optimizer_updates"] = int(model.trainer.ema.updates)
        if metadata["optimizer_updates"] < 1:
            raise RuntimeError("Training completed without an optimizer update")
        runtime = unwrap_model(model.trainer.model)
        if hasattr(runtime, "teacher_model"):
            metadata["teacher_state_unchanged"] = _state_hash(runtime.teacher_model) == metadata["initial_teacher_state_sha256"]
            metadata["teacher_frozen"] = all(not p.requires_grad and p.grad is None for p in runtime.teacher_model.parameters())
            if not metadata["teacher_state_unchanged"] or not metadata["teacher_frozen"]:
                raise RuntimeError("Teacher parameters/buffers changed during KD training")
        if config["device"] != "cpu":
            metadata["peak_cuda_allocated_gib"] = torch.cuda.max_memory_allocated() / 2**30
            metadata["peak_cuda_reserved_gib"] = torch.cuda.max_memory_reserved() / 2**30
        if Path(model.trainer.save_dir).resolve() != run_dir.resolve():
            raise RuntimeError("Ultralytics changed the experiment output directory")
        for relative in ("weights/best.pt", "results.csv", "args.yaml"):
            if not (run_dir / relative).is_file():
                raise RuntimeError(f"Missing mandatory training output {relative}")
        metadata.update(recorder.verify(run_dir / "results.csv"))
        saved_args = yaml.safe_load((run_dir / "args.yaml").read_text(encoding="utf-8"))
        for key in config["training"]:
            if saved_args[key] != config["training"][key]:
                raise RuntimeError(f"Training protocol changed: {key}")
        metrics = evaluate_checkpoint(run_dir / "weights" / "best.pt", data_yaml, run_dir, config)
        # Hash seluruh file lagi; byte identik tidak perlu di-decode dan divalidasi ulang.
        verify_dataset_unchanged(audit)
        summary = {"status": "completed", "run_dir": str(run_dir),
                   "best_pt": str(run_dir / "weights" / "best.pt"), "metrics": metrics,
                   "mode": config["mode"], "nonfinal": config["mode"] == "smoke",
                   "best_epoch": recorder.best_epoch, "test_evaluated": False}
        metadata.update(status="completed", total_seconds=time.perf_counter() - started,
                        artifact_upload_seconds=tracker.upload_seconds,
                        wandb_url=tracker.run.url if tracker.run is not None else None)
        write_json(run_dir / "metadata.json", metadata)
        write_json(run_dir / "run_summary.json", summary)
        tracker.finish("completed", metrics=metrics)
        metadata.update(total_seconds=time.perf_counter() - started, artifact_upload_seconds=tracker.upload_seconds)
        write_json(run_dir / "metadata.json", metadata)
        write_json(output / "latest_run.json", summary)
        return summary
    except BaseException as exc:
        error = f"{type(exc).__name__}: {exc}"
        key = os.getenv("WANDB_API_KEY")
        if key:
            error = error.replace(key, "[REDACTED]")
        metadata.update(status="failed", error=error, total_seconds=time.perf_counter() - started)
        write_json(run_dir / "metadata.json", metadata)
        write_json(run_dir / "run_summary.json", {"status": "failed", "run_dir": str(run_dir), "error": error})
        tracker.finish("failed", error=error)
        if isinstance(exc, torch.OutOfMemoryError):
            raise RuntimeError("GPU OOM at configured batch/imgsz; protocol was not automatically changed") from exc
        raise

In [ ]:
# Satu pemanggilan = satu eksperimen, tanpa loop seed/model.
SUMMARY = run_experiment(CONFIG)

## Ringkasan dan Unduh Hasil
Unduh best.pt dan CSV epoch. Kaggle Save Version menyimpan folder output. ZIP opsional lewat CREATE_RESULTS_ZIP=True. Gunakan teacher best.pt yang sama sebagai Input seluruh KD full. Colab: unduh hasil sebelum session berakhir.

In [ ]:
RUN_DIR = Path(SUMMARY["run_dir"])
print(f"Best epoch {SUMMARY['best_epoch']} | best.pt: {SUMMARY['best_pt']}", flush=True)
print("mAP50-95:", SUMMARY["metrics"]["mAP50_95"], flush=True)
if EVALUATE_COCO_AREA:
    print("COCO area:", SUMMARY["metrics"]["coco_area"], flush=True)
RESULT_ZIP = create_results_zip(RUN_DIR, OUTPUT_ROOT) if CREATE_RESULTS_ZIP else None
try:
    from IPython.display import display, FileLink
    for result_path in (RUN_DIR / "weights" / "best.pt", RUN_DIR / "epoch_history.csv", RUN_DIR / "checkpoint_manifest.csv", RESULT_ZIP):
        if result_path is not None:
            display(FileLink(str(result_path)))
except ImportError:
    print("Folder hasil:", RUN_DIR, flush=True)

# Opsional Colab: uncomment untuk langsung mengunduh ZIP.
# from google.colab import files
# if RESULT_ZIP is not None:
#     files.download(str(RESULT_ZIP))